# Tools技術探索

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例  

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-12b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

## 測試模型

In [2]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("簡短的說明機器學習的定義")
]
result = llm.invoke(messages)
print(result.content)

機器學習是一種人工智慧技術，讓電腦系統能夠從資料中「學習」，自動識別模式並做出預測或決策，而無需被明確地程式設計。


# 準備工具(Tools Function)

In [15]:
# 匯入 geopy 套件中的 Nominatim 類別，用於地理編碼（將地名轉換成經緯度）
from geopy.geocoders import Nominatim, ArcGIS

# 定義函式：輸入城市名稱，回傳其經緯度座標
def get_coordinates(city_name):
    """取得城市GPS座標，先用 Nominatim，失敗時改用 ArcGIS"""

    # ---- 第一來源：Nominatim ----
    try:
        geolocator1 = Nominatim(user_agent="clement_fallback_test")
        location = geolocator1.geocode(city_name, timeout=10)

        if location:
            return (location.latitude, location.longitude)
    except:
        pass  # 忽略錯誤，直接進入第二來源

    # ---- 第二來源：ArcGIS  ----
    try:
        geolocator2 = ArcGIS(timeout=10)
        location = geolocator2.geocode(city_name)

        if location:
            return (location.latitude, location.longitude)
    except Exception as e:
        return e

    return None

# 範例測試：查詢「台北」的經緯度
get_coordinates("台北")

(25.0375198, 121.5636796)

In [16]:
# 匯入 requests 套件，用於發送 HTTP 請求
import requests

# 定義函式：輸入經緯度，回傳目前天氣資訊（溫度）
def get_weather(latitude, longitude):
    """取得溫度值

    Args:
        latitude: GPS經度
        longitude: GPS緯度
    """    

    # 使用 Open-Meteo API 發送 GET 請求，取得氣象資料
    # API 參數：
    # - latitude / longitude: 經緯度
    # - current: 取得目前時刻的溫度 (temperature_2m) 與風速 (wind_speed_10m)
    # - hourly: 取得每小時溫度、相對濕度、風速
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}&"
        f"current=temperature_2m,wind_speed_10m&"
        f"hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    
    # 將回傳的 JSON 資料解析成 Python 字典
    data = response.json()
    
    # 回傳目前時刻的氣溫（單位：攝氏度）
    return data['current']['temperature_2m']

# 範例測試：取得台北市 (經緯度 25.0375198, 121.5636796) 的目前溫度
get_weather(25.0375198, 121.5636796)


22.6

# 將 Python 函式轉換為 LangChain Tools

In [17]:
from langchain_core.tools import tool # <-- 這裡是新增的

# 匯入 geopy 套件中的 Nominatim 類別，用於地理編碼（將地名轉換成經緯度）
from geopy.geocoders import Nominatim

# 定義函式：輸入城市名稱，回傳其經緯度座標
@tool # <-- 這裡是新增的
def tool_get_coordinates(city_name:str):
    """取得城市GPS座標，先用 Nominatim，失敗時改用 ArcGIS"""

    # ---- 第一來源：Nominatim ----
    try:
        geolocator1 = Nominatim(user_agent="clement_fallback_test")
        location = geolocator1.geocode(city_name, timeout=10)

        if location:
            return (location.latitude, location.longitude)
    except:
        pass  # 忽略錯誤，直接進入第二來源

    # ---- 第二來源：ArcGIS  ----
    try:
        geolocator2 = ArcGIS(timeout=10)
        location = geolocator2.geocode(city_name)

        if location:
            return (location.latitude, location.longitude)
    except Exception as e:
        return e

    return None

# 匯入 requests 套件，用於發送 HTTP 請求
import requests

# 定義函式：輸入經緯度，回傳目前天氣資訊（溫度）
@tool # <-- 這裡是新增的
def tool_get_weather(latitude:float, longitude:float):
    """取得溫度值

    Args:
        latitude: GPS經度
        longitude: GPS緯度
    """    

    # 使用 Open-Meteo API 發送 GET 請求，取得氣象資料
    # API 參數：
    # - latitude / longitude: 經緯度
    # - current: 取得目前時刻的溫度 (temperature_2m) 與風速 (wind_speed_10m)
    # - hourly: 取得每小時溫度、相對濕度、風速
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}&"
        f"current=temperature_2m,wind_speed_10m&"
        f"hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    
    # 將回傳的 JSON 資料解析成 Python 字典
    data = response.json()
    
    # 回傳目前時刻的氣溫（單位：攝氏度）
    return data['current']['temperature_2m']

In [19]:
get_coordinates

<function __main__.get_coordinates(city_name)>

In [18]:
tool_get_coordinates

StructuredTool(name='tool_get_coordinates', description='取得城市的GPS座標\n\nArgs:\n    city_name: 城市名稱', args_schema=<class 'langchain_core.utils.pydantic.tool_get_coordinates'>, func=<function tool_get_coordinates at 0x000002907FA49DA0>)

In [20]:
tool_get_coordinates.invoke({"city_name":"台北"})

(25.0375198, 121.5636796)

In [26]:
from langchain_core.tools import StructuredTool
tool_get_coordinates_v2 = StructuredTool.from_function(get_coordinates)
tool_get_coordinates_v2

StructuredTool(name='get_coordinates', description='取得城市的GPS座標\n\nArgs:\n    city_name: 城市名稱', args_schema=<class 'langchain_core.utils.pydantic.get_coordinates'>, func=<function get_coordinates at 0x000002907FA49EE0>)

# 連結Tools跟LLM

In [28]:
# 工具列表
tools = [tool_get_coordinates, tool_get_weather]

# 連結工具到LLM
llm_with_tools = llm.bind_tools(tools)

In [45]:
from langchain_core.messages import HumanMessage

# 使用者輸入訊息，詢問台北市目前的溫度
messages = [
    HumanMessage("台北市現在的溫度？")
]

# 透過 invoke() 呼叫 LLM，並傳入訊息列表
response_01 = llm_with_tools.invoke(messages)

# 查看 LLM 的回應
response_01


AIMessage(content='', additional_kwargs={'function_call': {'name': 'tool_get_coordinates', 'arguments': '{"city_name": "\\u53f0\\u5317\\u5e02"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--68b19ecd-b993-4f39-9855-a5ebe9963bd0-0', tool_calls=[{'name': 'tool_get_coordinates', 'args': {'city_name': '台北市'}, 'id': '94a9e24d-4e8d-4c70-a947-c581c07e6c9d', 'type': 'tool_call'}], usage_metadata={'input_tokens': 125, 'output_tokens': 160, 'total_tokens': 285, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 140}})

In [46]:
response_01.tool_calls

[{'name': 'tool_get_coordinates',
  'args': {'city_name': '台北市'},
  'id': '71008034-688c-42cb-96d4-a849e2be2795',
  'type': 'tool_call'}]

In [47]:
# 取得 response_01 中的第一個工具呼叫資訊（tool_calls 是一個 List）
tools_call = response_01.tool_calls[0]

# 使用 globals() 取得全域命名空間中對應工具名稱的函式物件
# tools_call['name'] 取得工具名稱字串，例如 'tool_get_coordinates'
# globals()[tools_call['name']] 會回傳該名稱所對應的函式物件本身
globals()[tools_call['name']]

StructuredTool(name='tool_get_coordinates', description='取得城市的GPS座標\n\nArgs:\n    city_name: 城市名稱', args_schema=<class 'langchain_core.utils.pydantic.tool_get_coordinates'>, func=<function tool_get_coordinates at 0x000002907FA49DA0>)

In [48]:
tool_result = globals()[tools_call['name']].invoke(tools_call['args'])
tool_result

(25.0375198, 121.5636796)

In [49]:
# 匯入 ToolMessage 類別，用來將工具執行結果包裝成訊息回傳給 LLM
from langchain_core.messages import ToolMessage

# 建立 ToolMessage 物件
tool_message = ToolMessage(
    content=tool_result,          # 工具執行後的結果，例如經緯度或氣溫
    name=tools_call["name"],      # 工具名稱，用來告訴 LLM 這筆訊息來自哪個工具
    tool_call_id=tools_call["id"] # 對應這次工具呼叫的唯一 ID，方便追蹤
)

# 查看 ToolMessage 物件內容
tool_message


ToolMessage(content=['25.0375198', '121.5636796'], name='tool_get_coordinates', tool_call_id='94a9e24d-4e8d-4c70-a947-c581c07e6c9d')

In [51]:
# 建立新的訊息列表（message list）
messages = [
    HumanMessage("台北市現在的溫度？"),  # 使用者第一輪問題
    response_01,                        # LLM 第一輪回應（包含工具呼叫資訊）
    tool_message                         # 工具執行結果，回傳給 LLM
]

# 透過 invoke() 呼叫 LLM，將完整訊息列表傳入
# LLM 將會依據使用者問題、先前回應與工具結果生成新的回覆
response_02 = llm_with_tools.invoke(messages)

# 查看 LLM 第二輪生成的最終回應
response_02


AIMessage(content='', additional_kwargs={'function_call': {'name': 'tool_get_weather', 'arguments': '{"latitude": "25.0375198", "longitude": "121.5636796"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--58cb8db1-2951-4935-a180-4f2b7dc3adeb-0', tool_calls=[{'name': 'tool_get_weather', 'args': {'latitude': '25.0375198', 'longitude': '121.5636796'}, 'id': '2109d36c-0eb4-4d6f-b630-8b254b348530', 'type': 'tool_call'}], usage_metadata={'input_tokens': 185, 'output_tokens': 41, 'total_tokens': 226, 'input_token_details': {'cache_read': 0}})

In [54]:
tools_call = response_02.tool_calls[0]
tool_result = globals()[tools_call['name']].invoke(tools_call['args'])
tool_result

22.6

In [56]:
# 建立 ToolMessage 物件
tool_message_02 = ToolMessage(
    content=tool_result,          # 工具執行後的結果，例如經緯度或氣溫
    name=tools_call["name"],      # 工具名稱，用來告訴 LLM 這筆訊息來自哪個工具
    tool_call_id=tools_call["id"] # 對應這次工具呼叫的唯一 ID，方便追蹤
)

# 建立新的訊息列表（message list）
messages = [
    HumanMessage("台北市現在的溫度？"),  # 使用者第一輪問題
    response_01,                        # LLM 第一輪回應（包含工具呼叫資訊）
    tool_message,                         # 第一輪工具執行結果，回傳給 LLM
    response_02,                        # LLM 第二輪回應
    tool_message_02,                         # 第二輪工具執行結果，回傳給 LLM    
]

response_03 = llm_with_tools.invoke(messages)

# 查看 LLM 第三輪生成的最終回應
response_03


AIMessage(content='台北市現在的溫度是 22.6°C。', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--21c094c3-34a4-4831-b872-aa9ed7e071c5-0', usage_metadata={'input_tokens': 246, 'output_tokens': 14, 'total_tokens': 260, 'input_token_details': {'cache_read': 0}})